In [8]:
import os
import re
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA, SequentialChain, TransformChain, LLMChain
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings

load_dotenv()

True

# Connect to ChromaDB

In [2]:
api_key = os.getenv("API_KEY")
llm = model = ChatGoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    temperature=0.0,
    max_tokens=50000,
    timeout=None,
    max_retries=2
)

collection_name = "langchain_docs_index"
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

vectorstore = Chroma(embedding_function=embedding, collection_name=collection_name, persist_directory="./data/vectors/chroma_db")

# Initialize retriever

In [3]:
retriever = vectorstore.as_retriever(search_kwargs={"k":2})

# Utility chains

Utility chains are pre-build for speceficic tasks. Examples:
- RetrievalQA
- AnalyzeDocumentChain

In [4]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

In [9]:
qa_chain.invoke("what is a Morpheme?")

{'query': 'what is a Morpheme?',
 'result': 'A morpheme is the smallest meaningful unit of language. It cannot be divided into smaller meaningful parts.\n\nFor example:\n*   The word "fox" consists of one morpheme: "fox".\n*   The word "cats" consists of two morphemes: "cat" (the root) and "-s" (an affix indicating plural).'}

In [14]:
qa_chain.invoke("what does BPE mean?")

{'query': 'what does BPE mean?',
 'result': 'BPE stands for Byte Pair Encoding.'}

# Foundational chains

Foundational chains can be:
- TransformChain: for transforming the data
- LLMChain: for using an llm with a prompt template -> Deprecated: Use RunnableSequence, e.g. ``prompt | llm`` instead

In [10]:
def limpiar_texto(entradas: dict) -> dict:
    texto = entradas["texto"]

    # Eliminamos los emojis utilizando un amplio rango unicode
    # Ten en cuenta que esto podría potencialmente eliminar algunos caracteres válidos que no son en inglés
    patron_emoji = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticonos
        "\U0001F300-\U0001F5FF"  # símbolos y pictogramas
        "\U0001F680-\U0001F6FF"  # símbolos de transporte y mapas
        "\U0001F1E0-\U0001F1FF"  # banderas (iOS)
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE,
    )
    texto = patron_emoji.sub(r'', texto)

    # Removemos las URLs
    patron_url = re.compile(r'https?://\S+|www\.\S+')
    texto = patron_url.sub(r'', texto)

    return {"texto_limpio": texto}

In [11]:
cadena_que_limpia = TransformChain(
    input_variables=["texto"],
    output_variables=["texto_limpio"],
    transform=limpiar_texto
)

In [13]:
cadena_que_limpia.invoke('Check this page https://twitter.com/home 🙈')

{'texto': 'Check this page https://twitter.com/home 🙈',
 'texto_limpio': 'Check this page  '}

In [14]:
plantilla = """Parafrasea este texto:

{texto_limpio}

En el estilo de una persona informal de {estilo}.

Parafraseado: """

prompt = PromptTemplate(
    input_variables=["texto_limpio", "estilo"],
    template=plantilla
)

In [19]:
cadena_que_cambia_estilo = prompt | llm

In [20]:
cadena_que_cambia_estilo.invoke(input={
    "texto_limpio": "Hola, ¿qué tal se encuentra usted?",
    "estilo": "Argentino",
})

AIMessage(content='¡Che, ¿cómo andás?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019efe87-24c6-7821-83ea-78321f279b99-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 8, 'total_tokens': 45, 'input_token_details': {'cache_read': 0}})